In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


In [ ]:
data=pd.read_csv("dataset1.csv")

In [ ]:
data.head()

,N,P,K,pH,EC,OC,S,Zn,Fe,Cu,Mn,B,Output
0,138,8.6,560,7.46,0.62,0.70,5.9,0.24,0.31,0.77,8.71,0.11,0
1,213,7.5,338,7.62,0.75,1.06,25.4,0.30,0.86,1.54,2.89,2.29,0
2,163,9.6,718,7.59,0.51,1.11,14.3,0.30,0.86,1.57,2.70,2.03,0
3,157,6.8,475,7.64,0.58,0.94,26.0,0.34,0.54,1.53,2.65,1.82,0
4,270,9.9,444,7.63,0.40,0.86,11.8,0.25,0.76,1.69,2.43,2.26,1


In [ ]:
data.isnull().sum()

,0
N,0
P,0
K,0
pH,0
EC,0
OC,0
S,0
Zn,0
Fe,0
Cu,0


In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 880 entries, 0 to 879
Data columns (total 13 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   N       880 non-null    int64  
 1   P       880 non-null    float64
 2   K       880 non-null    int64  
 3   pH      880 non-null    float64
 4   EC      880 non-null    float64
 5   OC      880 non-null    float64
 6   S       880 non-null    float64
 7   Zn      880 non-null    float64
 8   Fe      880 non-null    float64
 9   Cu      880 non-null    float64
 10  Mn      880 non-null    float64
 11  B       880 non-null    float64
 12  Output  880 non-null    int64  
dtypes: float64(10), int64(3)
memory usage: 89.5 KB


In [ ]:
from sklearn.model_selection import train_test_split

# Define features (X) and target (y)
X = data.drop('Output', axis=1)
y = data['Output']

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (880, 12)
Shape of y: (880,)


In [ ]:
print(X.describe())

               N           P           K          pH          EC          OC  \
count  880.00000  880.000000  880.000000  880.000000  880.000000  880.000000   
mean   246.73750   14.562159  499.978409    7.510500    0.543659    0.617989   
std     77.38886   21.967755  124.222838    0.464912    0.141597    0.842986   
min      6.00000    2.900000   11.000000    0.900000    0.100000    0.100000   
25%    201.00000    6.800000  412.000000    7.350000    0.430000    0.380000   
50%    257.00000    8.100000  475.000000    7.500000    0.545000    0.590000   
75%    307.00000   10.550000  581.000000    7.630000    0.640000    0.780000   
max    383.00000  125.000000  887.000000   11.150000    0.950000   24.000000   

                S          Zn          Fe          Cu          Mn           B  
count  880.000000  880.000000  880.000000  880.000000  880.000000  880.000000  
mean     7.545080    0.469273    4.140523    0.952443    8.666500    0.590159  
std      4.424184    1.894234    3.1100

In [ ]:
print(y.value_counts())

Output
1    440
0    401
2     39
Name: count, dtype: int64


In [ ]:
# Perform train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (704, 12)
X_test shape: (176, 12)
y_train shape: (704,)
y_test shape: (176,)


### Stratified K-Fold Cross-Validation with SMOTE in Each Fold

In [ ]:
# Install imbalanced-learn if you haven't already
!pip install imbalanced-learn -q

# Import necessary libraries
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

In [ ]:
# Initialize StratifiedKFold
stratified_kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Lists to store metrics for each fold
accuracies = []
precisions = []
recalls = []
f1_scores = []

# Iterate over each fold
for fold, (train_index, val_index) in enumerate(stratified_kfold.split(X, y)):
    print(f"--- Fold {fold+1}/{stratified_kfold.n_splits} ---")

    # Split data into training and validation sets for the current fold
    X_train_fold, X_val_fold = X.iloc[train_index], X.iloc[val_index]
    y_train_fold, y_val_fold = y.iloc[train_index], y.iloc[val_index]

    # Apply SMOTE to the training data of the current fold ONLY
    sm = SMOTE(random_state=42)
    X_train_res_fold, y_train_res_fold = sm.fit_resample(X_train_fold, y_train_fold)

    print(f"  Original training set shape for fold {fold+1}: {X_train_fold.shape}, {y_train_fold.shape}")
    print(f"  Resampled training set shape for fold {fold+1}: {X_train_res_fold.shape}, {y_train_res_fold.shape}")
    print(f"  New class distribution in resampled training set for fold {fold+1}:\n", y_train_res_fold.value_counts())

    # Initialize and train a Random Forest Classifier on the SMOTEd training data
    rf_model_fold = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_model_fold.fit(X_train_res_fold, y_train_res_fold)

    # Make predictions on the untouched validation fold
    y_pred_fold = rf_model_fold.predict(X_val_fold)

    # Evaluate the model for the current fold
    fold_accuracy = accuracy_score(y_val_fold, y_pred_fold)
    # Use 'weighted' average for precision, recall, f1_score with multi-class and imbalance
    fold_precision = precision_score(y_val_fold, y_pred_fold, average='weighted', zero_division=0)
    fold_recall = recall_score(y_val_fold, y_pred_fold, average='weighted', zero_division=0)
    fold_f1 = f1_score(y_val_fold, y_pred_fold, average='weighted', zero_division=0)

    accuracies.append(fold_accuracy)
    precisions.append(fold_precision)
    recalls.append(fold_recall)
    f1_scores.append(fold_f1)

    print(f"  Fold {fold+1} Accuracy: {fold_accuracy:.4f}")
    print(f"  Fold {fold+1} Precision: {fold_precision:.4f}")
    print(f"  Fold {fold+1} Recall: {fold_recall:.4f}")
    print(f"  Fold {fold+1} F1-score: {fold_f1:.4f}\n")


# Calculate and print average metrics across all folds
print("--- Average Metrics Across All Folds ---")
print(f"Mean Accuracy: {np.mean(accuracies):.4f} (Std: {np.std(accuracies):.4f})")
print(f"Mean Precision: {np.mean(precisions):.4f} (Std: {np.std(precisions):.4f})")
print(f"Mean Recall: {np.mean(recalls):.4f} (Std: {np.std(recalls):.4f})")
print(f"Mean F1-score: {np.mean(f1_scores):.4f} (Std: {np.std(f1_scores):.4f})")

print("\nIndividual Fold Accuracies:", [f'{acc:.4f}' for acc in accuracies])
print("Individual Fold Precisions:", [f'{prec:.4f}' for prec in precisions])
print("Individual Fold Recalls:", [f'{rec:.4f}' for rec in recalls])
print("Individual Fold F1-scores:", [f'{f1:.4f}' for f1 in f1_scores])

--- Fold 1/5 ---
  Original training set shape for fold 1: (704, 12), (704,)
  Resampled training set shape for fold 1: (1056, 12), (1056,)
  New class distribution in resampled training set for fold 1:
 Output
0    352
1    352
2    352
Name: count, dtype: int64
  Fold 1 Accuracy: 0.8807
  Fold 1 Precision: 0.8713
  Fold 1 Recall: 0.8807
  Fold 1 F1-score: 0.8732

--- Fold 2/5 ---
  Original training set shape for fold 2: (704, 12), (704,)
  Resampled training set shape for fold 2: (1056, 12), (1056,)
  New class distribution in resampled training set for fold 2:
 Output
0    352
1    352
2    352
Name: count, dtype: int64
  Fold 2 Accuracy: 0.8920
  Fold 2 Precision: 0.8860
  Fold 2 Recall: 0.8920
  Fold 2 F1-score: 0.8886

--- Fold 3/5 ---
  Original training set shape for fold 3: (704, 12), (704,)
  Resampled training set shape for fold 3: (1056, 12), (1056,)
  New class distribution in resampled training set for fold 3:
 Output
0    352
1    352
2    352
Name: count, dtype: int64


In [ ]:
import joblib

In [ ]:
# Apply SMOTE to the entire dataset (X, y) to prepare the final training data
sm = SMOTE(random_state=42)
X_res, y_res = sm.fit_resample(X, y)

print("Original dataset shape:", X.shape, y.shape)
print("Resampled dataset shape:", X_res.shape, y_res.shape)
print("New class distribution in resampled dataset:\n", y_res.value_counts())

Original dataset shape: (880, 12) (880,)
Resampled dataset shape: (1320, 12) (1320,)
New class distribution in resampled dataset:
 Output
0    440
1    440
2    440
Name: count, dtype: int64


In [ ]:
# Train the final Random Forest model on the full, SMOTE-resampled dataset
final_rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
final_rf_model.fit(X_res, y_res)

RandomForestClassifier(random_state=42)

In [ ]:
import numpy as np

# Create a sample array with the provided values
# sample_input_class_1 = np.array([[90, 42, 90, 6.5, 0.5, 0.8, 14, 0.8, 5.5, 0.5, 3.0, 0.6]])
sample_input_class_1 = np.array([[390, 42, 90, 6.5, 0.5, 0.8, 14, 0.8, 5.5, 0.5, 3.0, 0.6]])
# sample_input_class_1 = np.array([[390, 42, 90, 0.5, 0.5, 0.8, 7, 0.8, 5.5, 0.5, 3.0, 0.6]])

# Predict the class for the sample input
predicted_class = final_rf_model.predict(sample_input_class_1)

print(f"The predicted class for the sample input is: {predicted_class[0]}")

The predicted class for the sample input is: 1


In [ ]:
train_predictions = final_rf_model.predict(X_res)

print(np.unique(train_predictions, return_counts=True))

(array([0, 1, 2]), array([440, 440, 440]))


In [ ]:
print(final_rf_model.predict_proba(sample_input_class_1))

[[0.02 0.59 0.39]]


In [ ]:
# Save the trained model to a pickle file
model_filename = 'final_random_forest_model.pkl'
joblib.dump(final_rf_model, model_filename)

print(f"Model successfully saved as '{model_filename}'")